In [12]:
import Gmsh: gmsh
using Gridap, GridapGmsh

In [13]:
function create_square_model(h)
    gmsh.initialize()
    gmsh.model.add("unit_square");
    gmsh.model.geo.addPoint(0.0, 0.0, 0.0, h, 1); # last argument is optional identifier, unique per dimension
    gmsh.model.geo.addPoint(1.0, 0.0, 0.0, h, 2);
    gmsh.model.geo.addPoint(1.0, 1.0, 0.0, h, 3);
    gmsh.model.geo.addPoint(0.0, 1.0, 0.0, h, 4);
    gmsh.model.geo.addLine(1, 2, 1); 
    gmsh.model.geo.addLine(2, 3, 2); # line 2 goes from point 2 to point 3
    gmsh.model.geo.addLine(3, 4, 3);
    gmsh.model.geo.addLine(4, 1, 4);
    
    gmsh.model.geo.addCurveLoop([1, 2, 3, 4], 1);

    gmsh.model.geo.addPlaneSurface([1], 1);

    gmsh.model.geo.synchronize();

    # Define physical groups without the string argument
    edges_tag = gmsh.model.addPhysicalGroup(1, [1, 2, 3, 4])   # edges
    corners_tag = gmsh.model.addPhysicalGroup(0, [1, 2, 3, 4]) # corners
    domain_tag = gmsh.model.addPhysicalGroup(2, [1])           # surface
    
    # Set names for the physical groups
    gmsh.model.setPhysicalName(1, edges_tag, "boundary")
    gmsh.model.setPhysicalName(0, corners_tag, "boundary")
    gmsh.model.setPhysicalName(2, domain_tag, "domain")

    gmsh.model.mesh.generate(2); # dimension is 2
    
    model = GmshDiscreteModel(gmsh);
    gmsh.finalize();
    return model
end


create_square_model (generic function with 1 method)

In [14]:
h = 0.01; # target mesh size
model = create_square_model(h) # fix above function above using the tutorial
order = 1
reffe = ReferenceFE(lagrangian, Float64, order)
V = TestFESpace(model, reffe, conformity=:H1, dirichlet_tags="boundary")
U = TrialFESpace(V, 0.0)


Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 30%] Meshing curve 2 (Line)
Info    : [ 60%] Meshing curve 3 (Line)
Info    : [ 80%] Meshing curve 4 (Line)
Info    : Done meshing 1D (Wall 0.000550665s, CPU 0.000736s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.35269s, CPU 0.352256s)
Info    : 11833 nodes 23668 elements


TrialFESpace()

In [15]:
Ω = Triangulation(model)
dΩ = Measure(Ω, 2 * order)

GenericMeasure()

In [16]:
n=7
Δt, T = 2.0^(-n), 2*π/3 # time step, final time
num_steps = Int(round(T / Δt))

268

In [17]:
f(t, x) = (2*π*π - 4)* sin(π*x[1])* sin(π*x[2]) * cos(2*t) + (0.3*13*π*π - 0.3) * sin(2*π*x[1])* sin(3*π*x[2]) * sin(t)
f(t) = x -> f(t,x)

f (generic function with 2 methods)

In [18]:
u_exact(t, x) = sin(π*x[1])* sin(π*x[2]) * cos(2*t) +0.3*sin(2*π*x[1])* sin(3*π*x[2]) * sin(t)

# Initial data (for the PDE)
u₀ = x -> sin(π*x[1])* sin(π*x[2])
u₁ = x -> 0.3*sin(2*π*x[1])* sin(3*π*x[2])

#18 (generic function with 1 method)

In [19]:
m(u, v) = ∫(u*v)dΩ
k(u, v) = ∫(∇(u) ⋅ ∇(v))dΩ

M = assemble_matrix(m, U, V)
K = assemble_matrix(k, U, V)

# LHS matrix for (13.5):
A = (1/Δt^2)*M + (1
    /4)*K

11433×11433 SparseArrays.SparseMatrixCSC{Float64, Int64} with 79231 stored entries:
⎡⢻⣶⣴⣆⠀⣖⣦⡆⣢⠀⣀⣶⠖⣤⣆⠂⠀⠄⢴⠠⠀⢐⠜⣻⣟⡟⢿⣛⣻⣿⣿⣿⣧⡖⢧⣵⣵⠤⡟⠛⎤
⎢⠰⢿⢿⣷⣠⣶⣷⣄⣵⡀⣯⣿⢱⡩⡗⣃⡜⢘⣿⢶⡃⡄⠁⠿⡿⣿⣷⣿⣿⣿⣿⣿⣿⣿⣻⣿⣿⣿⠕⠀⎥
⎢⢠⢤⢠⣾⣿⣿⣿⡷⣀⢀⣾⣿⣼⡧⣯⠸⡄⢬⣿⢀⡎⢠⡀⢿⡿⣺⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣧⡦⠀⎥
⎢⠨⠿⠙⢿⢿⡿⣿⣿⣿⣆⣿⣻⢾⡂⣻⠙⡆⢼⠻⢐⠈⡌⡄⠿⣯⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⡫⠀⎥
⎢⠈⠚⠑⠻⠀⢘⠻⢿⣿⣿⡟⣿⣿⣷⣿⣸⣅⢬⢿⢰⠉⠚⠂⣻⢟⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣟⣥⠀⎥
⎢⢠⣼⣯⣿⣾⣿⣿⣻⣿⣭⣻⣾⣿⣿⣿⣿⡷⢻⣿⣀⢜⣪⡀⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣯⣿⡍⠀⎥
⎢⠘⣥⡕⡲⠶⡿⠺⠳⢿⣿⣿⣿⣿⣿⣿⣿⣿⣾⣿⣿⣟⣿⡃⣾⣿⣿⣷⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣯⢄⎥
⎢⠨⠙⠽⢩⣋⡛⣟⠚⣛⣻⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⡦⣽⣯⣿⣿⣟⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⡯⠠⎥
⎢⠀⠄⣒⢉⡀⣍⣈⣍⡁⣝⣽⣋⣻⣿⣿⣿⣿⣿⣿⣿⣿⣿⡷⣿⣿⣿⣿⣽⣿⣿⣿⣿⣿⣿⣿⣿⡿⣿⡷⠾⎥
⎢⠐⡓⢻⣟⠛⢛⢛⢂⢛⣓⠛⢻⣿⣿⣿⣿⣿⣿⣿⣿⡿⣿⣿⣿⣿⣿⣟⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⡿⠼⎥
⎢⢀⢀⠉⠬⠊⣉⡂⠤⣣⠀⡲⣱⣿⣽⣿⣿⣿⣿⣿⣯⣻⣾⣽⣿⣿⣿⣿⣿⣾⣿⣿⣿⣿⣿⣿⣿⣿⣿⣟⠇⎥
⎢⣶⣡⣥⡄⣤⣌⣤⡍⣬⣠⣤⣬⣩⣬⣌⣯⣽⣯⣿⣿⣷⣿⣻⣾⣿⣿⣿⣿⣿⣿⣽⣿⣿⣿⣿⣿⣿⣽⣿⣿⎥
⎢⣿⠽⣿⣯⣻⣫⣯⣿⣿⣵⣿⣿⣿⣿⣯⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⣿⢳⣽⣿⣿⣿⣿⣿⣿⣿⣿⣿⣽⣿⣿⢿⣟⣿⣿⣽⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣻⣿⎥
⎢⣿⣾⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣾⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣷⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⡿⎥
⎢⢩⠿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣻⎥
⎢⢍⣷⣿⣾⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣟⣿⣯⡙⎥
⎢⠑⡟⣿⣿⠿⣿⣿⣿⣿⢿⣯⣿⣿⣿⣿⣿⣿⣯⣿⣿⣿⣿⣟⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣽⣿⣿⡿⠫⎥
⎣⣿⠉⠑⠁⠈⠋⠋⠊⠁⠛⠃⠉⠋⢟⠋⡋⣹⡏⣛⡏⠿⠝⣿⣿⣿⣿⣿⣾⣿⣿⣿⡿⣿⣻⣏⠻⡿⡋⢿⣷⎦

In [26]:
uh0 = interpolate_everywhere(u₀, U)   # U^0
uh1 = interpolate_everywhere(u₁, U)   # U^1

SingleFieldFEFunction():
 num_cells: 23264
 DomainStyle: ReferenceDomain()
 Triangulation: BodyFittedTriangulation()
 Triangulation id: 3107876725797981772

In [27]:
if !isdir("tmp") mkdir("tmp") end

createpvd("wave") do pvd
    # spara initiala fält
    pvd[0.0] = createvtk(Ω, "tmp/wave3_0.vtu",
                         cellfields=["uh" => uh0, "e" => x -> 0.0])
    pvd[Δt]  = createvtk(Ω, "tmp/wave3_1.vtu",
                         cellfields=["uh" => uh1, "e" => (x -> u_exact(Δt,x)) - uh1])

    # jobba på dof-vektorer (enklast för att bygga RHS med M,K)
    uvec_prev = get_free_dof_values(uh0)  # U^{n-1}
    uvec_curr = get_free_dof_values(uh1)  # U^{n}

    e = nothing
    el2 = nothing

    # Vi kommer räkna fram U^{n+1} från n=1,...,num_steps-1
    for n in 1:(num_steps-1)
        t_n = n*Δt

        # Lastvektor F^n = ∫ f(t_n)*v dΩ
        b(v) = ∫( f(t_n)*v )dΩ
        Fn = assemble_vector(b, V)

        # RHS från (13.5) i matrisform:
        rhs = Fn +
              ((2/Δt^2)*M - (1/2)*K) * uvec_curr +
              (-(1/Δt^2)*M - (1/4)*K) * uvec_prev

        # lös för nästa steg
        uvec_next = A \ rhs
        uh_next = FEFunction(U, uvec_next)

        # fel
        t_next = (n+1)*Δt
        uex = x -> u_exact(t_next, x)
        e = uex - uh_next

        # spara
        vtufilename = "tmp/wave3_$(n+1).vtu"
        pvd[t_next] = createvtk(Ω, vtufilename, cellfields=["uh" => uh_next, "e" => e])

        # uppdatera
        uvec_prev = uvec_curr
        uvec_curr = uvec_next
    end

    # sista felet (från sista iterationen)
    el2 = sqrt(sum( ∫(e*e)*dΩ ))
    println("L2 error norm: $el2")
end

L2 error norm: 2.426381536014341


270-element Vector{String}:
 "wave.pvd"
 "tmp/wave3_0.vtu"
 "tmp/wave3_1.vtu"
 "tmp/wave3_2.vtu"
 "tmp/wave3_3.vtu"
 "tmp/wave3_4.vtu"
 "tmp/wave3_5.vtu"
 "tmp/wave3_6.vtu"
 "tmp/wave3_7.vtu"
 "tmp/wave3_8.vtu"
 "tmp/wave3_9.vtu"
 "tmp/wave3_10.vtu"
 "tmp/wave3_11.vtu"
 ⋮
 "tmp/wave3_257.vtu"
 "tmp/wave3_258.vtu"
 "tmp/wave3_259.vtu"
 "tmp/wave3_260.vtu"
 "tmp/wave3_261.vtu"
 "tmp/wave3_262.vtu"
 "tmp/wave3_263.vtu"
 "tmp/wave3_264.vtu"
 "tmp/wave3_265.vtu"
 "tmp/wave3_266.vtu"
 "tmp/wave3_267.vtu"
 "tmp/wave3_268.vtu"